# 06 · Análisis exploratorio: variables de Clara sobre la ventana de la consigna

**Etapa 9 del plan.** Prueba las dos ideas más prometedoras del protocolo de Clara, entrenando con 201903–06:
- **`ratio3`:** cuánto se aparta el mes actual del promedio propio del cliente;
- **`zero_streak3`:** cuántos meses seguidos lleva en cero.

**Todo lo de este notebook es exploratorio.** 201909 y 201910 ya se miraron, así que nada de acá demuestra una hipótesis ni cambia el resultado de la regla pre-registrada. Las comparaciones y el criterio de lectura se fijaron en la ficha, antes de ver estos resultados.

**No escribe en `exp/03/`, `exp/05/` ni en los `resultados/03_*`, `04_*` y `05_*`:** solo los lee.

Ficha: `fichas/etapa_9_exploratorio_clara.md` (en el laboratorio).

## 1. Parámetros

- **Retomar:** si la corrida se corta, se vuelve a correr entera. Las semillas ya hechas no se repiten.
- **Corrida de prueba:** `N_SEMILLAS = 2`, en un directorio aparte.

In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import platform
import time
import warnings

import duckdb
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter
from scipy import stats
from sklearn.metrics import roc_auc_score

# --- Parámetros -------------------------------------------------------------
BASE_PARQUET = Path('../data/base_2019.parquet')
RESULTADOS = Path('../resultados')
EXP_03 = Path('../exp/03')                          # solo se lee
EXP = Path('../exp/06')
FIGURAS = RESULTADOS / 'figuras'

CONTROL = 'AC'
CONFIGURACIONES = {                                 # configuración: variables nuevas
    'AC':    {'ratio': False, 'racha': False},
    'AC_R':  {'ratio': True,  'racha': False},
    'AC_Z':  {'ratio': False, 'racha': True},
    'AC_RZ': {'ratio': True,  'racha': True},
}
COMPARAR = ['AC_R', 'AC_Z', 'AC_RZ']                # cada una contra AC
REFERENCIA_03 = 'A-consigna'                        # los mismos modelos que AC

MESES_TRAIN = [201903, 201904, 201905, 201906]
MES_VALID = 201908
MESES_TEST = [201909, 201910]
MESES_EVAL = [MES_VALID] + MESES_TEST
ROTOS_EXTRA = [('ctarjeta_master_descuentos', 201910), ('mtarjeta_master_descuentos', 201910)]   # decisión 1 (b)

SEMILLA_PRIMIGENIA = 979201
N_SEMILLAS = 10
N_BOOTSTRAP = 2_000
NIVEL_IC = 0.95
BANDA = (3_000, 6_000)                              # ambos incluidos
EPSILON_RATIO = 1e-12
POSITIVOS = ['BAJA+1', 'BAJA+2']
N_HILOS = 8

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO = 27_500
VENTANA_SUAVIZADO = 501
LARGO_CURVA = 25_000
# -----------------------------------------------------------------------------

PRED_03 = EXP_03 / 'pred'
PRED = EXP / 'pred'
HUELLAS = EXP / 'huellas.json'
for d in (PRED, FIGURAS):
    d.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings('ignore', message='.*ChainedAssignmentError.*')  # falso positivo pandas 2.2 + Py 3.14

ESPERADO_FILAS_POR_MES = {
    201901: 123_916, 201902: 125_020, 201903: 125_622, 201904: 126_116, 201905: 126_763,
    201906: 128_370, 201907: 129_853, 201908: 131_775, 201909: 133_419, 201910: 135_633,
}
ESPERADO_EVAL = {201908: (131_775, 543), 201909: (133_419, 615), 201910: (135_633, 632)}   # filas, BAJA+2
ESPERADO_N_VARIABLES = {'AC': 760, 'AC_R': 889, 'AC_Z': 889, 'AC_RZ': 1_018}
ESPERADO_ROTOS = 23


def primos_6_digitos():
    criba = np.ones(1_000_000, dtype=bool)
    criba[:2] = False
    for i in range(2, 1_001):
        if criba[i]:
            criba[i * i::i] = False
    return np.flatnonzero(criba[100_000:]) + 100_000


TODAS = np.random.default_rng(SEMILLA_PRIMIGENIA).choice(primos_6_digitos(), size=20, replace=False).tolist()[:10]
assert TODAS == [294103, 342929, 161093, 677561, 431077, 330679, 511579, 312773, 118147, 567529]
SEMILLAS = TODAS[:N_SEMILLAS]


def huella_archivos():
    """Control 8: huella de las salidas anteriores, para comprobar que no cambian."""
    return {p.name: hashlib.blake2b(p.read_bytes(), digest_size=16).hexdigest()
            for p in sorted(RESULTADOS.glob('0[345]_*'))}


HUELLA_INICIAL = huella_archivos()
T_INICIO = time.time()
print('Salidas anteriores registradas:', len(HUELLA_INICIAL))

## 2. Funciones y carga

Las mismas definiciones del `03` al `05`. Los meses rotos pasan a NA antes de calcular nada: los 23 de siempre **más las 2 variables de Master rotas en 201910** (decisión 1).

In [ ]:
def curva(prob, es_b2):
    orden = np.argsort(-prob, kind='stable')
    return np.cumsum(np.where(es_b2[orden], GANANCIA_ACIERTO, -COSTO_ESTIMULO))[:LARGO_CURVA].astype(np.float64)


def elegir_n(curva_media):
    suave = pd.Series(curva_media).rolling(VENTANA_SUAVIZADO, center=True, min_periods=VENTANA_SUAVIZADO).mean().to_numpy()
    return int(np.nanargmax(suave)) + 1


def banda(cv):
    return cv[..., BANDA[0] - 1:BANDA[1]].mean(axis=-1)


def wilcoxon_exacto(d):
    d = np.round(np.asarray(d, dtype=np.float64), 6)
    d = d[d != 0]
    n = len(d)
    if n == 0:
        return 1.0
    rangos = stats.rankdata(np.abs(d))
    w = float((np.sign(d) * rangos).sum())
    signos = 1 - 2 * ((np.arange(2 ** n)[:, None] >> np.arange(n)) & 1)
    return float(np.mean(np.abs(signos @ rangos) >= abs(w) - 1e-9))


def ic(x):
    x = np.asarray(x, dtype=np.float64)
    media = x.mean()
    medio = stats.t.ppf((1 + NIVEL_IC) / 2, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))
    return media, media - medio, media + medio


assert abs(wilcoxon_exacto(np.arange(1, 11)) - 2 / 1024) < 1e-12
assert abs(wilcoxon_exacto([1, 1, 1, 1, 1, 1, 1, 1, 1, -1]) - 22 / 1024) < 1e-12

HIPER = json.loads((RESULTADOS / '03_hiperparametros.json').read_text())
N_03 = pd.read_csv(RESULTADOS / '03_N.csv').set_index('config')['N'].astype(int).to_dict()
ganancias_04 = pd.read_csv(RESULTADOS / '04_ganancias.csv')
claves_npz = np.load(EXP_03 / 'claves_eval.npz')
CLIENTE_EVAL, MES = claves_npz['numero_de_cliente'], claves_npz['foto_mes']
PROB_03 = np.stack([np.load(PRED_03 / REFERENCIA_03 / f'{s}.npy') for s in SEMILLAS])

t0 = time.time()
con = duckdb.connect()
base = con.execute(f"""
    select * from read_parquet('{BASE_PARQUET.as_posix()}')
    where foto_mes <= {max(MESES_TEST)}
    order by foto_mes, numero_de_cliente
""").df()
con.close()
assert base['foto_mes'].value_counts().sort_index().to_dict() == ESPERADO_FILAS_POR_MES

CLAVES = ['numero_de_cliente', 'foto_mes', 'clase_ternaria']
VARIABLES = [c for c in base.columns if c not in CLAVES]
BANDERAS = ['active_quarter', 'cliente_vip', 'internet', 'cdescubierto_preacordado',
            'tcallcenter', 'thomebanking', 'tmobile_app', 'ccajas_transacciones']
ESTADOS = ['tcuentas', 'Visa_status', 'Master_status', 'Visa_delinquency', 'Master_delinquency']
FECHAS = [f'{t}_{c}' for t in ('Visa', 'Master') for c in ('Fvencimiento', 'Finiciomora', 'fultimo_cierre', 'fechaalta')]
EDADES = ['cliente_edad', 'cliente_antiguedad']
MONTOS = [v for v in VARIABLES if v not in BANDERAS + ESTADOS + FECHAS + EDADES]
assert len(VARIABLES) == 152 and len(MONTOS) == 129

rotos = pd.read_csv(RESULTADOS / '01_meses_rotos.csv')[['variable', 'foto_mes']]
assert len(rotos) == ESPERADO_ROTOS
for v, mes in ROTOS_EXTRA:
    assert not ((rotos['variable'] == v) & (rotos['foto_mes'] == mes)).any(), v
    assert (base.loc[base['foto_mes'] == mes, v] == 0).all(), v      # de verdad están rotas
rotos = pd.concat([rotos, pd.DataFrame(ROTOS_EXTRA, columns=['variable', 'foto_mes'])], ignore_index=True)
for v, mes in rotos.itertuples(index=False):
    base.loc[base['foto_mes'] == mes, v] = np.nan

FOTO_MES = base['foto_mes'].to_numpy()
PERIODO = (FOTO_MES // 100 * 12 + FOTO_MES % 100).astype(np.int64)
CLAVE = base['numero_de_cliente'].to_numpy(np.int64) * 100_000 + PERIODO
INDICE = pd.Index(CLAVE)
assert INDICE.is_unique
X_MES = np.asfortranarray(base[VARIABLES].to_numpy(np.float32))
CLASE = base['clase_ternaria'].to_numpy()
clase_eval = base.loc[base['foto_mes'].isin(MESES_EVAL), ['numero_de_cliente', 'foto_mes', 'clase_ternaria']]
del base
gc.collect()

eval_ = pd.DataFrame({'numero_de_cliente': CLIENTE_EVAL, 'foto_mes': MES}).merge(
    clase_eval, on=['numero_de_cliente', 'foto_mes'], how='left', validate='one_to_one')
assert eval_['clase_ternaria'].notna().all()
ES_B2 = (eval_['clase_ternaria'] == 'BAJA+2').to_numpy()
es_mes = {m: MES == m for m in MESES_EVAL}
for mes, (filas, b2) in ESPERADO_EVAL.items():            # control 1
    assert (es_mes[mes].sum(), ES_B2[es_mes[mes]].sum()) == (filas, b2), mes
print(f'Dataset cargado en {time.time() - t0:.0f} s · {len(rotos)} casos de meses rotos a NA')

## 3. Tabla de variables

Las 760 columnas de A, calculadas como en el `03`, más las nuevas sobre los 129 montos y conteos, con la ventana t, t−1 y t−2:
- **`ratio3`:** `x_t / mean3`. NA si falta alguna foto o si `|mean3|` ≤ 1e-12;
- **`zero_streak3`:** meses consecutivos en cero contando desde t, de 0 a 3. NA si falta alguna foto.

Un mes roto ya es NA, así que tampoco vuelve por estas variables.

In [ ]:
def especificacion(config):
    ex = CONFIGURACIONES[config]
    e = [('original', v, 0) for v in VARIABLES]
    e += [(op, v, k) for k in (1, 2) for op in ('lag', 'delta') for v in VARIABLES]
    if ex['ratio']:
        e += [('ratio', v, 3) for v in MONTOS]
    if ex['racha']:
        e += [('zero_streak', v, 3) for v in MONTOS]
    return e


def nombre(op, v, k):
    return v if op == 'original' else f'{v}_{op}{k}'


def construir(config):
    filas = np.flatnonzero(np.isin(FOTO_MES, MESES_TRAIN + MESES_EVAL))
    mes_fila = FOTO_MES[filas]
    n_train = int(np.isin(mes_fila, MESES_TRAIN).sum())
    assert np.isin(mes_fila[:n_train], MESES_TRAIN).all() and np.isin(mes_fila[n_train:], MESES_EVAL).all()
    pos = {k: INDICE.get_indexer(CLAVE[filas] - k) for k in range(3)}

    espec = especificacion(config)
    ubic = {e: i for i, e in enumerate(espec)}
    F = np.empty((len(filas), len(espec)), dtype=np.float32)
    with np.errstate(invalid='ignore', divide='ignore'):
        for j, v in enumerate(VARIABLES):
            F[:, ubic[('original', v, 0)]] = X_MES[filas, j]
            hk = {}
            for k in range(3):
                col = X_MES[pos[k], j]
                col[pos[k] < 0] = np.nan
                hk[k] = col
            for k in (1, 2):
                F[:, ubic[('lag', v, k)]] = hk[k]
                F[:, ubic[('delta', v, k)]] = hk[0] - hk[k]
            completo = ~(np.isnan(hk[0]) | np.isnan(hk[1]) | np.isnan(hk[2]))
            if ('ratio', v, 3) in ubic:
                a, b, c = (hk[k].astype(np.float64) for k in range(3))
                media = (a + b + c) / 3
                F[:, ubic[('ratio', v, 3)]] = np.where(completo & (np.abs(media) > EPSILON_RATIO), a / media, np.nan)
            if ('zero_streak', v, 3) in ubic:
                racha = np.where(hk[0] != 0, 0, np.where(hk[1] != 0, 1, np.where(hk[2] != 0, 2, 3)))
                F[:, ubic[('zero_streak', v, 3)]] = np.where(completo, racha, np.nan)
    return {'F': F, 'columnas': [nombre(*e) for e in espec], 'filas': filas, 'mes_fila': mes_fila, 'n_train': n_train}


def huella_control(T):
    hsh = hashlib.blake2b(digest_size=16)
    hsh.update(T['filas'].tobytes())
    for j in range(ESPERADO_N_VARIABLES['AC']):
        hsh.update(T['F'][:, j].tobytes())
    return hsh.hexdigest()


def controlar(T, config):
    F, col, mes_fila = T['F'], {c: i for i, c in enumerate(T['columnas'])}, T['mes_fila']
    ex = CONFIGURACIONES[config]
    assert F.shape[1] == ESPERADO_N_VARIABLES[config], F.shape            # control 2
    evalua = slice(T['n_train'], None)
    assert (CLAVE[T['filas'][evalua]] // 100_000 == CLIENTE_EVAL).all() and (mes_fila[evalua] == MES).all()
    for v, mes in ROTOS_EXTRA:                                            # control 5
        assert np.isnan(F[mes_fila == mes, col[v]]).all(), v

    rng = np.random.default_rng(SEMILLA_PRIMIGENIA)                       # control 6: fórmulas a mano
    muestra = rng.choice(len(T['filas']), size=2_000, replace=False)
    for v in ('ctrx_quarter', 'cpayroll_trx'):
        valores = pd.Series(X_MES[:, VARIABLES.index(v)].astype(np.float64), index=INDICE)
        y = np.stack([valores.reindex(CLAVE[T['filas'][muestra]] - k).to_numpy() for k in range(3)])
        completo = ~np.isnan(y).any(axis=0)
        if ex['ratio']:
            media = y.mean(axis=0)
            usa = completo & (np.abs(media) > EPSILON_RATIO)
            np.testing.assert_allclose(F[muestra[usa], col[f'{v}_ratio3']], y[0, usa] / media[usa], rtol=1e-5)
            assert np.isnan(F[muestra[~usa], col[f'{v}_ratio3']]).all()
        if ex['racha']:
            racha = [next((k for k in range(3) if y[k, i] != 0), 3) for i in np.flatnonzero(completo)]
            np.testing.assert_array_equal(F[muestra[completo], col[f'{v}_zero_streak3']], racha)
            assert np.isnan(F[muestra[~completo], col[f'{v}_zero_streak3']]).all()

    periodo_fila = mes_fila // 100 * 12 + mes_fila % 100                  # control 7: meses rotos
    for v, mes in rotos.itertuples(index=False):
        if v in MONTOS:
            p = (mes // 100) * 12 + mes % 100
            for k in range(3):
                destino = periodo_fila == p + k
                if destino.any():
                    for sufijo, activo in (('ratio3', ex['ratio']), ('zero_streak3', ex['racha'])):
                        if activo:
                            assert np.isnan(F[destino, col[f'{v}_{sufijo}']]).all(), (v, mes, k)

## 4. Modelos

Hiperparámetros, árboles y semillas del `03`, en las 4 configuraciones (decisión 2).

**Control:** las primeras 760 columnas son idénticas en las 4 configuraciones.

In [ ]:
FIJOS = {'objective': 'binary', 'metric': 'auc', 'boosting': 'gbdt', 'max_bin': 31, 'bagging_freq': 1,
         'lambda_l1': 0.0, 'lambda_l2': 0.0, 'min_gain_to_split': 0.0, 'feature_pre_filter': False,
         'num_threads': N_HILOS, 'deterministic': True, 'force_col_wise': True, 'verbose': -1}
MEJORES = {**FIJOS, **HIPER['params']}
ARBOLES = HIPER['arboles']


def datos(T):
    n = T['n_train']
    y_train = np.isin(CLASE[T['filas'][:n]], POSITIVOS).astype(np.int8)
    return lgb.Dataset(T['F'][:n], label=y_train, feature_name=T['columnas'],
                       params={'max_bin': 31, 'feature_pre_filter': False}, free_raw_data=False)


huellas = json.loads(HUELLAS.read_text()) if HUELLAS.exists() else {}
segundos = {}
for config in CONFIGURACIONES:
    (PRED / config).mkdir(exist_ok=True)
    pendientes = [s for s in SEMILLAS if not (PRED / config / f'{s}.npy').exists()]
    if not pendientes:
        print(f'{config}: las {len(SEMILLAS)} semillas ya estaban')
        continue
    t0 = time.time()
    T = construir(config)
    controlar(T, config)
    huellas[config] = huella_control(T)
    HUELLAS.write_text(json.dumps(huellas, indent=2))
    assert len(set(huellas.values())) == 1, huellas       # control 3
    dtrain = datos(T)
    print(f'{config}: {T["n_train"]:,} filas de entrenamiento · {T["F"].shape[1]:,} variables · tabla y controles en {time.time() - t0:.0f} s')
    for semilla in pendientes:
        t0 = time.time()
        modelo = lgb.train({**MEJORES, 'seed': semilla}, dtrain, num_boost_round=ARBOLES)
        prob = modelo.predict(T['F'][T['n_train']:], num_threads=N_HILOS).astype(np.float32)
        np.save(PRED / config / f'{semilla}.npy', prob)
        segundos.setdefault(config, []).append(time.time() - t0)
        print(f'{config} · semilla {semilla}: {time.time() - t0:.0f} s')
    del T, dtrain, modelo, prob
    gc.collect()

faltan = [(c, s) for c in CONFIGURACIONES for s in SEMILLAS if not (PRED / c / f'{s}.npy').exists()]
assert not faltan, faltan                                 # control 9
PROB = {c: np.stack([np.load(PRED / c / f'{s}.npy') for s in SEMILLAS]) for c in CONFIGURACIONES}

## 5. Control de reproducción y efecto de las variables de Master

201910 no entra en el entrenamiento, así que `AC` son **los mismos modelos** que A-consigna del `03`. En 201908 y 201909 tienen que dar las mismas predicciones. Solo cambia 201910, y esa diferencia mide cuánto pesan las 2 variables de Master sin tratar, en esta ventana.

In [ ]:
misma_maquina = HIPER['maquina'] == platform.node()
no_octubre = es_mes[201908] | es_mes[201909]
dif_no_octubre = float(np.abs(PROB[CONTROL][:, no_octubre] - PROB_03[:, no_octubre]).max())
dif_octubre = float(np.abs(PROB[CONTROL][:, es_mes[201910]] - PROB_03[:, es_mes[201910]]).max())
print(f'AC contra A-consigna · diferencia máxima de probabilidad: 201908 y 201909 {dif_no_octubre:.2e} · 201910 {dif_octubre:.2e}')
if misma_maquina:
    assert dif_no_octubre < 1e-6                          # control 4

n_ac = elegir_n(np.stack([curva(p[es_mes[MES_VALID]], ES_B2[es_mes[MES_VALID]]) for p in PROB[CONTROL]]).mean(axis=0))
if N_SEMILLAS == 10:
    assert n_ac == N_03[REFERENCIA_03], (n_ac, N_03[REFERENCIA_03])

filas = []
m = 201910
for i, s in enumerate(SEMILLAS):
    antes, despues = (curva(p[i][es_mes[m]], ES_B2[es_mes[m]]) for p in (PROB_03, PROB[CONTROL]))
    esperado = ganancias_04.query('config == @REFERENCIA_03 and foto_mes == @m and semilla == @s')['ganancia_M'].item()
    if N_SEMILLAS == 10:
        assert abs(antes[n_ac - 1] / 1e6 - esperado) < 6e-5, (s, esperado)
    filas.append({'semilla': s, 'N': n_ac, 'ganancia_sin_tratar_M': antes[n_ac - 1] / 1e6, 'ganancia_tratada_M': despues[n_ac - 1] / 1e6,
                  'banda_sin_tratar_M': banda(antes) / 1e6, 'banda_tratada_M': banda(despues) / 1e6,
                  'auc_sin_tratar': roc_auc_score(ES_B2[es_mes[m]], PROB_03[i][es_mes[m]]),
                  'auc_tratada': roc_auc_score(ES_B2[es_mes[m]], PROB[CONTROL][i][es_mes[m]])})
master = pd.DataFrame(filas).assign(d_M=lambda t: t['ganancia_tratada_M'] - t['ganancia_sin_tratar_M'],
                                    d_banda_M=lambda t: t['banda_tratada_M'] - t['banda_sin_tratar_M'])
master.round(6).to_csv(RESULTADOS / '06_master.csv', index=False)
media, inf, sup = ic(master['d_M'])
print(f'EXPLORATORIO · efecto de tratar las 2 variables de Master en 201910: {media:+.2f} M (IC 95 %: {inf:.1f} a {sup:.1f}) · '
      f'banda {master["d_banda_M"].mean():+.2f} M · AUC {(master["auc_tratada"] - master["auc_sin_tratar"]).mean():+.5f}')

## 6. Comparaciones

Contra `AC`, en 201909 y 201910:
- **Por semilla:** N en la meseta de la curva media de 201908; IC 95 % con t de Student; Wilcoxon exacto (p descriptivo, sin corrección); banda.
- **Ensamble:** promedio de las probabilidades de las semillas, N elegido en 201908, bootstrap pareado de clientes.

In [ ]:
curvas = {(c, m): np.stack([curva(p[es_mes[m]], ES_B2[es_mes[m]]) for p in PROB[c]]) for c in CONFIGURACIONES for m in MESES_EVAL}
N = {c: elegir_n(curvas[(c, MES_VALID)].mean(axis=0)) for c in CONFIGURACIONES}
G = {k: v[:, N[k[0]] - 1] / 1e6 for k, v in curvas.items()}
B = {k: banda(v) / 1e6 for k, v in curvas.items()}

filas = []
for c in CONFIGURACIONES:
    for m in MESES_EVAL:
        for i, s in enumerate(SEMILLAS):
            filas.append({'config': c, 'semilla': s, 'foto_mes': m, 'N': N[c], 'ganancia_M': round(G[(c, m)][i], 4),
                          'ganancia_banda_M': round(B[(c, m)][i], 4),
                          'auc': round(roc_auc_score(ES_B2[es_mes[m]], PROB[c][i][es_mes[m]]), 6)})
semillas_tabla = pd.DataFrame(filas)
semillas_tabla.to_csv(RESULTADOS / '06_semillas.csv', index=False)

ENSAMBLE = {c: PROB[c].astype(np.float64).mean(axis=0) for c in CONFIGURACIONES}
CURVA_E = {(c, m): curva(ENSAMBLE[c][es_mes[m]], ES_B2[es_mes[m]]) for c in CONFIGURACIONES for m in MESES_EVAL}
N_E = {c: elegir_n(CURVA_E[(c, MES_VALID)]) for c in CONFIGURACIONES}


def aporte(prob, es_b2, n):
    """Lo que deja cada cliente del mes: 0 si no se lo contacta."""
    contactado = np.zeros(len(prob), dtype=bool)
    contactado[np.argsort(-prob, kind='stable')[:n]] = True
    return np.where(contactado, np.where(es_b2, GANANCIA_ACIERTO, -COSTO_ESTIMULO), 0).astype(np.float64)


def remuestrear(D, n_veces, semilla):
    rng = np.random.default_rng(semilla)
    n = D.shape[0]
    con_valor = np.flatnonzero(D.any(axis=1))
    salida = np.empty((n_veces, D.shape[1]))
    for b in range(n_veces):
        veces = np.bincount(rng.integers(0, n, size=n), minlength=n)
        salida[b] = veces[con_valor] @ D[con_valor]
    return salida


filas = []
for m in MESES_TEST:
    for c in COMPARAR:                                    # por semilla
        d = (G[(c, m)] - G[(CONTROL, m)]).round(6)
        media, inf, sup = ic(d)
        filas.append({'medida': 'semillas', 'config': c, 'contra': CONTROL, 'foto_mes': m, 'N': N[c], 'N_contra': N[CONTROL],
                      'd_M': media, 'ic95_inf_M': inf, 'ic95_sup_M': sup, 'a_favor': f'{int((d > 0).sum())}/{len(d)}',
                      'p_descriptivo': wilcoxon_exacto(d), 'd_banda_M': (B[(c, m)] - B[(CONTROL, m)]).mean()})
    b2 = ES_B2[es_mes[m]]                                 # ensamble
    base_c = aporte(ENSAMBLE[CONTROL][es_mes[m]], b2, N_E[CONTROL])
    assert base_c.sum() == CURVA_E[(CONTROL, m)][N_E[CONTROL] - 1]
    D = np.column_stack([aporte(ENSAMBLE[c][es_mes[m]], b2, N_E[c]) - base_c for c in COMPARAR])
    assert (remuestrear((base_c - base_c)[:, None], 20, SEMILLA_PRIMIGENIA) == 0).all()
    sumas = remuestrear(D, N_BOOTSTRAP, SEMILLA_PRIMIGENIA + m) / 1e6
    for j, c in enumerate(COMPARAR):
        filas.append({'medida': 'ensamble', 'config': c, 'contra': CONTROL, 'foto_mes': m, 'N': N_E[c], 'N_contra': N_E[CONTROL],
                      'd_M': D[:, j].sum() / 1e6, 'ic95_inf_M': np.percentile(sumas[:, j], 2.5),
                      'ic95_sup_M': np.percentile(sumas[:, j], 97.5),
                      'a_favor': f'{(sumas[:, j] > 0).mean() * 100:.1f} %',
                      'p_descriptivo': np.nan, 'd_banda_M': (banda(CURVA_E[(c, m)]) - banda(CURVA_E[(CONTROL, m)])) / 1e6})
comparacion = pd.DataFrame(filas)
comparacion.round(4).to_csv(RESULTADOS / '06_comparacion.csv', index=False)
print('EXPLORATORIO')
print(comparacion.round(3).to_string(index=False))

## 7. Figuras

Tamaño 1920 × 1080 (16:9). Todas llevan el rótulo de exploratorias.

In [ ]:
TINTA_2, GRILLA, AZUL = '#52514e', '#e1e0d9', '#2a78d6'
plt.rcParams.update({'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb', 'axes.edgecolor': '#c3c2b7',
                     'axes.labelcolor': TINTA_2, 'xtick.color': TINTA_2, 'ytick.color': TINTA_2, 'text.color': '#0b0b0b',
                     'axes.grid': True, 'grid.color': GRILLA, 'grid.linewidth': 0.6, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 13, 'axes.titlesize': 15})
TAMANO, DPI = (12.8, 7.2), 150


def coma(x, decimales=1):
    return f'{x:,.{decimales}f}'.replace(',', '_').replace('.', ',').replace('_', '.')


NOMBRES = {'AC_R': 'ratio3', 'AC_Z': 'zero_streak3', 'AC_RZ': 'las dos'}


def figura(medida, titulo, rotulo, archivo):
    fig, ejes = plt.subplots(1, 2, figsize=TAMANO, sharex=True)
    etiquetas = [NOMBRES[c] for c in COMPARAR]
    y = np.arange(len(etiquetas))[::-1]
    for ax, m in zip(ejes, MESES_TEST):
        t = comparacion[(comparacion['medida'] == medida) & (comparacion['foto_mes'] == m)].set_index('config').loc[COMPARAR]
        ax.axvline(0, color=TINTA_2, lw=1.2)
        ax.hlines(y, t['ic95_inf_M'], t['ic95_sup_M'], color=AZUL, lw=2)
        ax.scatter(t['d_M'], y, s=70, color=AZUL, zorder=3, edgecolor='#fcfcfb')
        ax.set_yticks(y, etiquetas)
        ax.set_ylim(-0.7, len(etiquetas) - 0.3)
        ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: coma(v, 0)))
        ax.set_title(str(m), loc='left')
        ax.set_xlabel(rotulo)
    fig.suptitle(titulo, x=0.02, ha='left', fontsize=17)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    fig.savefig(FIGURAS / archivo, dpi=DPI)
    plt.show()


figura('semillas', 'Exploratorio: variables de Clara contra A-consigna, por semilla',
       'Diferencia media (M$), IC 95 %', '06_1_por_semilla.png')
figura('ensamble', 'Exploratorio: variables de Clara contra A-consigna, ensamble',
       'Diferencia (M$), IC 95 % por bootstrap', '06_2_ensamble.png')

## 8. Criterio de lectura y cierre

Una variable es **candidata para la competencia 2** si cumple las tres (decisión 4, fijada antes de correr):
1. diferencia media por semilla positiva en 201909 y en 201910;
2. diferencia en la banda positiva en los dos meses, por semilla;
3. lo mismo en la banda del ensamble.

No es una prueba estadística: exige consistencia entre medidas. **Control:** las salidas anteriores no cambiaron.

In [ ]:
assert huella_archivos() == HUELLA_INICIAL, 'Cambiaron las salidas del 03, del 04 o del 05'   # control 8
print(f'Control de salidas OK · {(time.time() - T_INICIO) / 60:.1f} min en total')
if segundos:
    print('Segundos por modelo:', {c: round(float(np.mean(v)), 1) for c, v in segundos.items()})
print()
print('EXPLORATORIO: no cambia el resultado de la regla pre-registrada.')
print()
s = comparacion[comparacion['medida'] == 'semillas']
e = comparacion[comparacion['medida'] == 'ensamble']
candidatas = []
for c in COMPARAR:
    uno = (s.loc[s['config'] == c, 'd_M'] > 0).all()
    dos = (s.loc[s['config'] == c, 'd_banda_M'] > 0).all()
    tres = (e.loc[e['config'] == c, 'd_banda_M'] > 0).all()
    if uno and dos and tres:
        candidatas.append(c)
    print(f'{c} ({NOMBRES[c]}): media por semilla {"sí" if uno else "no"} · banda por semilla {"sí" if dos else "no"} · '
          f'banda del ensamble {"sí" if tres else "no"}')
print()
print('Candidatas para la competencia 2:', ', '.join(candidatas) if candidatas else 'ninguna')